# Chroma CRUD Operation

In [ ]:
import os
import shutil
from pathlib import Path
from uuid import uuid4

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings

## 1. Setup Paths and the Vector Stores

In [2]:
project_root = Path.cwd()
if project_root.name == "notebook":
    project_root = project_root.parent

project_root

WindowsPath('e:/Machine Learning and Data Science/Advanced-RAGs-Detailed/04 Vector Stores')

In [3]:
load_dotenv()

True

In [5]:
# Use a fixed collection name and persistence path so each rerun is predictable.
collection_name = "demo_2"
persist_directory = project_root  / "chroma_langchain_db"

print(f"Collection name: {collection_name}")
print(f"Persist directory: {persist_directory}")

Collection name: demo_2
Persist directory: e:\Machine Learning and Data Science\Advanced-RAGs-Detailed\04 Vector Stores\chroma_langchain_db


In [6]:
# Start fresh so the CRUD flow produces the same result each time.
if persist_directory.exists():
    shutil.rmtree(persist_directory)
    print("Removed the old Chroma directory.")
else:
    print("No previous Chroma directory was found.")

No previous Chroma directory was found.


### Vector Store

In [7]:
embeddings = GoogleGenerativeAIEmbeddings(model='gemini-embedding-001')

vector_store = Chroma(
    collection_name = collection_name,
    embedding_function = embeddings,
    persist_directory = str(persist_directory),
)

print("vector Store is ready")

vector Store is ready


#### Helper function

In [8]:
def preview_text(text, limit=80):
    """Return a short preview for cleaner notebook output."""
    if len(text) <= limit:
        return text
    return text[:limit] + "..."


def print_documents(title, docs):
    """Print Document objects in a beginner-friendly format."""
    print(title)
    for index, doc in enumerate(docs, start=1):
        print(f"{index}. id={doc.id}")
        print(f"   topic={doc.metadata.get('topic')} | doc_number={doc.metadata.get('doc_number')}")
        print(f"   content={doc.page_content}")
    print()

## 3. Create and Insert Example Documents

In [11]:
# Keep the raw sample data separate from the Document objects so it is easier to read.
document_examples = [
    {
        "topic": "AI",
        "doc_number": 1,
        "text": "Artificial intelligence helps machines perform tasks that usually need human reasoning.",
    },
    {
        "topic": "AI",
        "doc_number": 2,
        "text": "AI systems can analyze patterns in data to support predictions and automation.",
    },
    {
        "topic": "AI",
        "doc_number": 3,
        "text": "Responsible AI development includes fairness, transparency, and safety checks.",
    },
    {
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG combines retrieval with generation so the model can answer using external knowledge.",
    },
    {
        "topic": "RAG",
        "doc_number": 5,
        "text": "A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.",
    },
    {
        "topic": "RAG",
        "doc_number": 6,
        "text": "Vector stores are important in RAG because they make semantic search over embedded documents possible.",
    },
    {
        "topic": "LLM",
        "doc_number": 7,
        "text": "LLMs generate text by predicting likely next tokens from patterns learned during training.",
    },
    {
        "topic": "LLM",
        "doc_number": 8,
        "text": "Prompt design can improve how clearly an LLM follows instructions and returns useful answers.",
    },
    {
        "topic": "Cricket",
        "doc_number": 9,
        "text": "Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.",
    },
    {
        "topic": "Cricket",
        "doc_number": 10,
        "text": "A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.",
    },
]

print(f"Prepared {len(document_examples)} document examples.")

Prepared 10 document examples.


In [12]:
for doc in document_examples:
    print(doc)

{'topic': 'AI', 'doc_number': 1, 'text': 'Artificial intelligence helps machines perform tasks that usually need human reasoning.'}
{'topic': 'AI', 'doc_number': 2, 'text': 'AI systems can analyze patterns in data to support predictions and automation.'}
{'topic': 'AI', 'doc_number': 3, 'text': 'Responsible AI development includes fairness, transparency, and safety checks.'}
{'topic': 'RAG', 'doc_number': 4, 'text': 'RAG combines retrieval with generation so the model can answer using external knowledge.'}
{'topic': 'RAG', 'doc_number': 5, 'text': 'A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'}
{'topic': 'RAG', 'doc_number': 6, 'text': 'Vector stores are important in RAG because they make semantic search over embedded documents possible.'}
{'topic': 'LLM', 'doc_number': 7, 'text': 'LLMs generate text by predicting likely next tokens from patterns learned during training.'}
{'topic': 'LLM', 'doc_number': 8, 'text': 'Prompt design can

In [ ]:
# convert the sample data into Langchain Document Objects
documents = [
    Document(
        id=str(uuid4()),
        page_content=item["text"],
        metadata={"topic":item["topic"], "doc_number": item["doc_number"]}
    )
    for item in document_examples
]

print_documents("Dummy documents prepared:",documents)

Dummy documents prepared:
1. id=ffd4d9c7-fc9c-4505-854f-f05bf300230c
   topic=AI | doc_number=1
   content=Artificial intelligence helps machines perform tasks that usually need human reasoning.
2. id=6107edfe-70d6-458d-b57c-6e9a64a3dca5
   topic=AI | doc_number=2
   content=AI systems can analyze patterns in data to support predictions and automation.
3. id=fc7aa0dc-ba91-4278-8092-c6c339bf30dc
   topic=AI | doc_number=3
   content=Responsible AI development includes fairness, transparency, and safety checks.
4. id=8e8c53c5-38e5-4171-ac2b-91e94b6047fe
   topic=RAG | doc_number=4
   content=RAG combines retrieval with generation so the model can answer using external knowledge.
5. id=b2fb727b-07a6-4904-82be-fd0914413d78
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.
6. id=71ab92ee-a188-41c6-997a-383553f961a4
   topic=RAG | doc_number=6
   content=Vector stores are important in RAG because they mak

In [14]:
# Insert the documents into the vector store

document_ids = vector_store.add_documents(documents)

print("Inserted documents ids:")
for doc_id in document_ids:
    print(doc_id)

print(f"\nTotal inserted documents: {len(document_ids)}")

Inserted documents ids:
ffd4d9c7-fc9c-4505-854f-f05bf300230c
6107edfe-70d6-458d-b57c-6e9a64a3dca5
fc7aa0dc-ba91-4278-8092-c6c339bf30dc
8e8c53c5-38e5-4171-ac2b-91e94b6047fe
b2fb727b-07a6-4904-82be-fd0914413d78
71ab92ee-a188-41c6-997a-383553f961a4
e34ec240-ca52-4bdf-8624-ad66d4b456d0
591a5b6f-ed6d-40af-b38f-4a3779d587b2
be58964d-34d0-418c-97fe-7d2d9b6ae146
c4d79a19-3e04-4bf9-9d21-1cb721ac9908

Total inserted documents: 10


## 4. Read the Stored Data 

In [20]:
raw_records = vector_store.get(include=['embeddings','metadatas'])
raw_records.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])

In [23]:
print(raw_records['embeddings'].shape)

(10, 3072)


In [24]:
print(raw_records['embeddings'][0])

[-0.01103104  0.00782148  0.00854845 ... -0.0034849   0.00138828
  0.00087448]


In [25]:
print(f"Total records in collection: {len(raw_records['ids'])}")
print("First three ids from get():")
for doc_id in raw_records["ids"][:3]:
    print(doc_id)

Total records in collection: 10
First three ids from get():
ffd4d9c7-fc9c-4505-854f-f05bf300230c
6107edfe-70d6-458d-b57c-6e9a64a3dca5
fc7aa0dc-ba91-4278-8092-c6c339bf30dc


In [26]:
selected_ids = document_ids[-3:]
selected_ids

['591a5b6f-ed6d-40af-b38f-4a3779d587b2',
 'be58964d-34d0-418c-97fe-7d2d9b6ae146',
 'c4d79a19-3e04-4bf9-9d21-1cb721ac9908']

In [27]:
# get_by_ids() returns LangChain Document objects instead of the raw Chroma dictionary.
selected_documents = vector_store.get_by_ids(selected_ids)
print_documents("Documents fetched with get_by_ids():", selected_documents)

Documents fetched with get_by_ids():
1. id=591a5b6f-ed6d-40af-b38f-4a3779d587b2
   topic=LLM | doc_number=8
   content=Prompt design can improve how clearly an LLM follows instructions and returns useful answers.
2. id=be58964d-34d0-418c-97fe-7d2d9b6ae146
   topic=Cricket | doc_number=9
   content=Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.
3. id=c4d79a19-3e04-4bf9-9d21-1cb721ac9908
   topic=Cricket | doc_number=10
   content=A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.



## 5. Similarity Search

In [28]:
query = "How does RAG help an LLM answer questions using outside knowledge?"
query

'How does RAG help an LLM answer questions using outside knowledge?'

In [29]:
search_results = vector_store.similarity_search(query,k=3)
print(f"Query: {query}\n")
print_documents("Similarity search results:", search_results)

Query: How does RAG help an LLM answer questions using outside knowledge?

Similarity search results:
1. id=8e8c53c5-38e5-4171-ac2b-91e94b6047fe
   topic=RAG | doc_number=4
   content=RAG combines retrieval with generation so the model can answer using external knowledge.
2. id=b2fb727b-07a6-4904-82be-fd0914413d78
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.
3. id=71ab92ee-a188-41c6-997a-383553f961a4
   topic=RAG | doc_number=6
   content=Vector stores are important in RAG because they make semantic search over embedded documents possible.



In [31]:
vector_store.similarity_search_with_score(query,k=2)

[(Document(id='8e8c53c5-38e5-4171-ac2b-91e94b6047fe', metadata={'doc_number': 4, 'topic': 'RAG'}, page_content='RAG combines retrieval with generation so the model can answer using external knowledge.'),
  0.349837988615036),
 (Document(id='b2fb727b-07a6-4904-82be-fd0914413d78', metadata={'doc_number': 5, 'topic': 'RAG'}, page_content='A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'),
  0.45079097151756287)]

## 6. Update Existing Documents

In [32]:
ids_to_update = [document_ids[3], document_ids[7]]
ids_to_update

['8e8c53c5-38e5-4171-ac2b-91e94b6047fe',
 '591a5b6f-ed6d-40af-b38f-4a3779d587b2']

In [33]:
# Keep the replacement text separate so the update step stays easy to follow.
updated_examples = [
    {
        "id": ids_to_update[0],
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG improves answer quality by retrieving relevant context before the language model generates a response.",
    },
    {
        "id": ids_to_update[1],
        "topic": "LLM",
        "doc_number": 8,
        "text": "Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.",
    },
]

updated_documents = [
    Document(
        id=item["id"],
        page_content=item["text"],
        metadata={"topic": item["topic"], "doc_number": item["doc_number"]},
    )
    for item in updated_examples
]

print_documents("Updated document content:", updated_documents)

Updated document content:
1. id=8e8c53c5-38e5-4171-ac2b-91e94b6047fe
   topic=RAG | doc_number=4
   content=RAG improves answer quality by retrieving relevant context before the language model generates a response.
2. id=591a5b6f-ed6d-40af-b38f-4a3779d587b2
   topic=LLM | doc_number=8
   content=Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.



In [34]:
vector_store.update_documents(ids=ids_to_update, documents=updated_documents)

In [35]:
print("Updated these ids:")
for doc_id in ids_to_update:
    print(doc_id)

Updated these ids:
8e8c53c5-38e5-4171-ac2b-91e94b6047fe
591a5b6f-ed6d-40af-b38f-4a3779d587b2


In [36]:
# Read the updated records back from Chroma to confirm the new values were stored.
updated_raw_records = vector_store.get(ids=ids_to_update)

print("Raw records returned by get(ids=ids_to_update):")
for doc_id, document_text, metadata in zip(
    updated_raw_records["ids"],
    updated_raw_records["documents"],
    updated_raw_records["metadatas"],
):
    print(f"id={doc_id}")
    print(f"metadata={metadata}")
    print(f"content={preview_text(document_text)}")
    print()

Raw records returned by get(ids=ids_to_update):
id=8e8c53c5-38e5-4171-ac2b-91e94b6047fe
metadata={'topic': 'RAG', 'doc_number': 4}
content=RAG improves answer quality by retrieving relevant context before the language m...

id=591a5b6f-ed6d-40af-b38f-4a3779d587b2
metadata={'doc_number': 8, 'topic': 'LLM'}
content=Well-written prompts help an LLM stay focused, follow instructions, and produce ...



In [37]:
updated_query = "How can retrieved context improve an LLM response in RAG?"
updated_query

'How can retrieved context improve an LLM response in RAG?'

In [38]:
updated_search_results = vector_store.similarity_search(updated_query, k=2)
print(f"Updated query: {updated_query}\n")
print_documents("Similarity search after update:", updated_search_results)

Updated query: How can retrieved context improve an LLM response in RAG?

Similarity search after update:
1. id=8e8c53c5-38e5-4171-ac2b-91e94b6047fe
   topic=RAG | doc_number=4
   content=RAG improves answer quality by retrieving relevant context before the language model generates a response.
2. id=b2fb727b-07a6-4904-82be-fd0914413d78
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.



## 7. Delete Documents

In [39]:
ids_to_delete = [document_ids[8], document_ids[9]]
ids_to_delete

['be58964d-34d0-418c-97fe-7d2d9b6ae146',
 'c4d79a19-3e04-4bf9-9d21-1cb721ac9908']

In [40]:
vector_store.delete(ids=ids_to_delete)

print("Deleted these ids:")
for doc_id in ids_to_delete:
    print(doc_id)

Deleted these ids:
be58964d-34d0-418c-97fe-7d2d9b6ae146
c4d79a19-3e04-4bf9-9d21-1cb721ac9908


In [41]:
remaining_records = vector_store.get()
remaining_ids = remaining_records["ids"]

print(f"Remaining document count: {len(remaining_ids)}")
print("Remaining ids:")
for doc_id in remaining_ids:
    print(doc_id)

print("\nDeleted ids still present?")
for doc_id in ids_to_delete:
    print(f"{doc_id}: {doc_id in remaining_ids}")

Remaining document count: 8
Remaining ids:
ffd4d9c7-fc9c-4505-854f-f05bf300230c
6107edfe-70d6-458d-b57c-6e9a64a3dca5
fc7aa0dc-ba91-4278-8092-c6c339bf30dc
8e8c53c5-38e5-4171-ac2b-91e94b6047fe
b2fb727b-07a6-4904-82be-fd0914413d78
71ab92ee-a188-41c6-997a-383553f961a4
e34ec240-ca52-4bdf-8624-ad66d4b456d0
591a5b6f-ed6d-40af-b38f-4a3779d587b2

Deleted ids still present?
be58964d-34d0-418c-97fe-7d2d9b6ae146: False
c4d79a19-3e04-4bf9-9d21-1cb721ac9908: False


In [42]:
print([doc.metadata['topic'] for doc in documents if doc.id in remaining_ids])

['AI', 'AI', 'AI', 'RAG', 'RAG', 'RAG', 'LLM', 'LLM']
